# Evaluation

Load one published stage, measure it on the train and validation splits, print
greedy samples, and score the reference chat tasks.

Evaluation is forward only. This notebook writes no checkpoint. It writes one
JSON report and pushes it beside the stage folder.


## Setup

The setup is two cells (docs/dependency-layering-plan.md section 6.5): one
installs the distribution and one configures the environment. The install
carries the shared library, so a session that has the package clones nothing
and builds nothing. The notebook's own modules (`session`, `hub_checkpoint`)
live in the repository's `notebooks/` directory beside this notebook. When the
package is absent, `session.setup` falls back to the clone and the Bazel
build. See `docs/checkpoint-format.md` section 15.


In [ ]:
%pip install -q "nanochat-cpp[hub]"
# The install carries both import packages. D6 has not published the binary
# wheel of section 6.3 yet, so the shared library comes from the search order
# (docs/dependency-layering-plan.md section 6.3). When the wheel exists, pin
# the version here:
# %pip install -q "https://.../nanochat_cpp_cuda_sm75-0.1.0.whl"


In [ ]:
import os; os.environ.setdefault("NANOCHAT_SANDBOX_BACKEND", "none")


In [ ]:
from pathlib import Path

import session

sess = session.setup(arch="sm_75", devices=1)
session.describe(sess)

import hub_checkpoint
import nanochat_cpp as nc


## Load the stage

`hub_checkpoint.load_stage` downloads the stage and builds the model, the
tokenizer, and the model arguments. The checkpoint load is parameter only, so
the reader skips the pretraining optimizer section.


In [ ]:
RUN_NAME = "base-L16-H1024-n8-kv4-s4096"
REPO_ID = hub_checkpoint.repo_id_for(RUN_NAME)
STAGE = "base"          # "base" or "sft"
REVISION = "base-500"   # the tag the pretraining notebook created

model, tokenizer, model_args = hub_checkpoint.load_stage(
    REPO_ID, STAGE, sess.base / "hub", revision=REVISION, device="cuda")
config = model_args.to_config()
print(REPO_ID, STAGE, REVISION)
print("device:", model.device, "parameters:", f"{model.param_total():,}")


## Data

Download the held-out validation shard. The bits-per-byte loop reads the train
shard and the validation shard.


In [ ]:
import os

DATA = sess.base / "base_data_climbmix"
TRAIN_SHARD = hub_checkpoint.TRAIN_SHARD
VAL_SHARD = hub_checkpoint.VAL_SHARD

hub_checkpoint.download_climbmix(DATA, token=os.environ.get("HF_TOKEN"))


## Bits per byte

One `nc.Evaluator.bpb` call runs several steps. The cell splits the budget so
it can print progress. `EVAL_CHUNKS = 1` runs one atomic call.


In [ ]:
EVAL_TOKENS = 40 * 524288
EVAL_BATCH = 8
EVAL_CHUNKS = 8   # progress lines per split; 1 runs one atomic call


def base_bpb_for(parquet, split):
    data = nc.TokenData(parquet=str(parquet), tokenizer=tokenizer,
                        seq_len=config.seq_len, batch=EVAL_BATCH, seed=44)
    evaluator = nc.Evaluator(model)
    total = max(1, EVAL_TOKENS // (EVAL_BATCH * config.seq_len))
    chunks = max(1, min(EVAL_CHUNKS, total))
    per_chunk = total // chunks
    values = []
    for index in range(chunks):
        steps = per_chunk if index < chunks - 1 else total - per_chunk * index
        if steps <= 0:
            continue
        values.append(evaluator.bpb(data, steps))
        done = per_chunk * (index + 1)
        print(f"  {split}: {done}/{total} steps  "
              f"bpb ~{sum(values) / len(values):.4f}", flush=True)
    bpb = sum(values) / len(values) if values else float("nan")
    print(f"{STAGE} {split:>5} bpb: {bpb:.4f}", flush=True)
    return bpb


base_bpb = {"train": base_bpb_for(DATA / TRAIN_SHARD, "train"),
            "val": base_bpb_for(DATA / VAL_SHARD, "val")}
print(f"{STAGE} validation bits per byte: {base_bpb['val']:.4f}", flush=True)


## Samples

Greedy completions of a fixed prompt set.


In [ ]:
import kaggle_demo

kaggle_demo.print_samples(model, tokenizer)


## Chat tasks

`nc.chat.ChatEvaluator` scores the reference tasks with the loaded tokenizer.
A task that fails becomes a skipped report, so one missing dataset does not
stop the run.


In [ ]:
EVAL_TASKS = ["ARC-Easy", "ARC-Challenge", "MMLU", "GSM8K", "HumanEval"]
chat_report = nc.chat.ChatEvaluator(model, tokenizer=tokenizer).run(
    tasks=EVAL_TASKS, max_problems=20, seed=0)
for task in chat_report.tasks:
    status = "skipped" if task.skipped else f"{task.accuracy:.3f}"
    print(f"{task.name:>14}  {status:>8}  ({task.correct}/{task.total})")
print("categorical core:", chat_report.categorical_core)
print("chat core:", chat_report.chat_core)


## Report

Write one JSON report and push it beside the stage folder.


In [ ]:
import json

report_data = {
    "stage": STAGE,
    "revision": REVISION,
    "repo_id": REPO_ID,
    "run_name": RUN_NAME,
    # The checkout's revision, or the installed distribution's, and the
    # session mode that says which (section 6.5).
    "git_revision": sess.revision,
    "session_mode": sess.mode,
    "bits_per_byte": base_bpb,
    "chat": [
        {"task": task.name, "accuracy": task.accuracy,
         "correct": task.correct, "total": task.total,
         "skipped": task.skipped}
        for task in chat_report.tasks
    ],
    "created_utc": hub_checkpoint.now_utc(),
}
report_name = f"reports/{STAGE}-{REVISION}.json"
report_path = sess.base / "reports" / f"{STAGE}-{REVISION}.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report_data, indent=1, sort_keys=True) + "\n",
                       encoding="utf-8")
hub_checkpoint.push_file(REPO_ID, report_name, report_path,
                         token=os.environ.get("HF_TOKEN"))
print("report:", report_name)


## Next

Set `STAGE = "sft"` and `REVISION` to the `sft` tag to measure the fine-tuned
stage. The load path is the same.
